# Lab Sheet 12: AI Chatbot Development
Hybrid rule-based + AI-generated chatbot (Google Colab + Gemini)

**Before running:** get a free key at aistudio.google.com → *Get API key*. In Colab, click the 🔑 *Secrets* icon on the left, add a secret named `GEMINI_API_KEY`, and turn on *Notebook access*. Then run the cells in order.

In [1]:
# Lab Sheet 12: AI Chatbot Development
# Hybrid Rule-Based and AI-Generated Chatbot (Google Colab + Gemini)
# Before running: add your key in Colab Secrets (key icon on the left)
# as GEMINI_API_KEY and turn on "Notebook access".

import os
import re
import requests

# ---------------- Setup ----------------
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    API_KEY = os.getenv("GEMINI_API_KEY")

MODEL = "gemini-flash-lite-latest"
API_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"
# Store recent conversation context.
conversation_history = []

print("API key loaded:", bool(API_KEY))


API key loaded: True


In [2]:
# Predefined chatbot rules
def rule_based_response(message):
    message = message.lower().strip()
    words = set(re.findall(r"[a-z]+", message))  # whole words only

    if words & {"hello", "hi", "hey"}:
        return "Hello! Welcome to the AI chatbot. How can I help you?"
    elif "what is ai" in message:
        return ("Artificial Intelligence is a field of computer science "
                "that enables machines to perform tasks requiring "
                "human-like intelligence.")
    elif "what is python" in message:
        return ("Python is a high-level programming language used in "
                "AI, automation, web development, and data science.")
    elif "your name" in message:
        return "I am a hybrid chatbot developed using Python."
    elif message in ["bye", "exit", "quit"]:
        return "Goodbye! Have a nice day."
    return None


In [3]:
# Contextual AI response
def ai_response(message):
    global conversation_history

    if not API_KEY:
        return ("No rule matched, and the AI API key is not configured. "
                "Add GEMINI_API_KEY in Colab Secrets to enable AI responses.")

    # Include the last 6 messages for context.
    history_text = "\n".join(
        f"{item['role']}: {item['content']}"
        for item in conversation_history[-6:]
    )

    prompt = (
        "You are a helpful educational chatbot. "
        "Answer clearly and concisely.\n\n"
        f"Recent conversation:\n{history_text}\n\n"
        f"User: {message}"
    )

    headers = {
        "x-goog-api-key": API_KEY,
        "Content-Type": "application/json"
    }
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 250}
    }

    try:
        response = requests.post(API_URL, headers=headers, json=payload, timeout=30)
        if response.status_code != 200:
            return f"AI service error {response.status_code}: {response.text[:300]}"

        result = response.json()
        parts = result["candidates"][0]["content"].get("parts", [])
        answer = "".join(part.get("text", "") for part in parts).strip()

        if not answer:
            return "The AI service returned no text response."

        # Keep a short history for follow-up questions.
        conversation_history.append({"role": "user", "content": message})
        conversation_history.append({"role": "assistant", "content": answer})
        return answer
    except (requests.RequestException, KeyError, IndexError) as error:
        return f"Unable to contact the AI service: {error}"

r = requests.get("https://generativelanguage.googleapis.com/v1beta/models",
                 headers={"x-goog-api-key": API_KEY}, timeout=30)
for m in r.json().get("models", []):
    if "generateContent" in m.get("supportedGenerationMethods", []):
        print(m["name"].replace("models/", ""))

print(API_URL)

gemini-2.5-flash
gemini-2.5-pro
gemini-2.5-flash-preview-tts
gemini-2.5-pro-preview-tts
gemma-4-26b-a4b-it
gemma-4-31b-it
gemini-flash-latest
gemini-flash-lite-latest
gemini-pro-latest
gemini-2.5-flash-lite
gemini-2.5-flash-image
gemini-3-flash-preview
gemini-3.1-pro-preview
gemini-3.1-pro-preview-customtools
gemini-3.1-flash-lite-preview
gemini-3.1-flash-lite
gemini-3-pro-image-preview
gemini-3-pro-image
nano-banana-pro-preview
gemini-3.1-flash-image-preview
gemini-3.1-flash-image
gemini-3.1-flash-lite-image
gemini-nano-banana-2.1
gemini-3.5-flash
gemini-3.5-flash-lite
gemini-omni-flash-preview
gemini-omni-1.1-flash
gemini-omni-1.2-flash
gemini-3.5-transcribe
gemini-3.6-flash
gemini-3.7-flash
gemini-3.8-flash
gemini-4-argon
gemini-4-carbon
gemini-4-hydrogen
gemini-4-neon
lyria-3-clip-preview
lyria-3-pro-preview
lyria-3.5
gemini-3.1-flash-tts-preview
gemini-3.8-flash-tts
gemini-3.8-flash-lite-tts
gemini-robotics-er-2-preview
gemini-2.5-computer-use-preview-10-2025
antigravity-preview-0

In [ ]:
# Main chatbot loop: type in the box that appears below; 'bye' to exit
def chatbot():
    print("=" * 55)
    print("Welcome to the Hybrid AI Chatbot")
    print("Type 'bye' to exit.")
    print("=" * 55)
    while True:
        user_input = input("\nYou: ").strip()
        if not user_input:
            print("Bot: Please enter a message.")
            continue

        # Try predefined rules first.
        answer = rule_based_response(user_input)
        if answer is not None:
            print("Bot:", answer)
            if user_input.lower() in ["bye", "exit", "quit"]:
                break
        else:
            # Use AI for queries not covered by rules.
            print("Bot:", ai_response(user_input))


chatbot()


Welcome to the Hybrid AI Chatbot
Type 'bye' to exit.

You: what is ai
Bot: Artificial Intelligence is a field of computer science that enables machines to perform tasks requiring human-like intelligence.

You: what is samsung
Bot: Samsung is a major multinational manufacturing conglomerate based in South Korea. It is best known for its global electronics business—producing smartphones (like the Galaxy series), televisions, and home appliances—but the company also operates in heavy industries, shipbuilding, insurance, and construction.


## Observation table
Runs the six test cases from the lab sheet and labels each answer as rule-based or AI-generated.

In [ ]:
# Observation table: run the six test cases automatically
conversation_history.clear()
test_cases = ["Hello", "What is AI?", "What is Python?",
              "Explain machine learning", "Give another example", "Bye"]

for i, text in enumerate(test_cases, 1):
    answer = rule_based_response(text)
    source = "Rule-based"
    if answer is None:
        answer = ai_response(text)
        source = "AI-generated"
    print(f"Test {i} | Input: {text}\n  [{source}] {answer}\n")
